# Is the Evidence Strong Enough to Build the Feature?

**Scenario:** A digital health app for cardiology patients is considering an adherence-support feature built around medication reminders and refill nudges. The idea comes from an existing observation that people who use statins appear to have better health outcomes.

Before committing engineering time to the feature, a useful question to ask would be:

**Among people who are supposed to be taking statins, would improving their adherence improve outcomes enough to justify building the feature?**

The problem is that statin users are not a random group of people. They may differ from non-users in age, healthcare use, health behaviors, and other factors that also affect health. So a difference between the two groups does not necessarily tell us what statin use itself is doing.

I used NHANES data to work through that problem. I first looked at the raw difference between statin users and non-users, then used propensity score matching to make the groups more comparable on the characteristics we can observe in this observational dataset.




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

pd.set_option('display.max_columns', 50)

BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/'

## 1. Load and merge the data

I first identified current statin users and compared them with non-users.

The outcome is self-reported general health, coded as good health versus fair/poor health. I also brought in several characteristics that could help explain differences between the two groups: age, sex, income, physical activity, smoking history, and healthcare use.


In [ ]:
demo = pd.read_sas(BASE + 'DEMO_I.XPT', format='xport')[['SEQN','RIDAGEYR','RIAGENDR','INDFMPIR']]

rxq = pd.read_sas(BASE + 'RXQ_RX_I.XPT', format='xport')[['SEQN','RXDDRGID']]
# RXQ_DRUG is the consolidated NHANES drug-information file linked by RXDDRGID.
rxq_drug = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/1988/DataFiles/RXQ_DRUG.XPT', format='xport')[['RXDDRGID','RXDDRUG']]
rxq = rxq.merge(rxq_drug, on='RXDDRGID', how='left')

STATINS = ['ATORVASTATIN','SIMVASTATIN','ROSUVASTATIN','PRAVASTATIN','LOVASTATIN','FLUVASTATIN','PITAVASTATIN']
rxq['is_statin'] = rxq['RXDDRUG'].astype(str).str.upper().str.contains('|'.join(STATINS), na=False)
statin_flag = rxq.groupby('SEQN')['is_statin'].any().reset_index()
statin_flag.columns = ['SEQN','statin_user']

hsq = pd.read_sas(BASE + 'HSQ_I.XPT', format='xport')[['SEQN','HSD010']]
# HSD010: 1=Excellent .. 5=Poor. Dichotomize: 1-3 = good health, 4-5 = fair/poor.
hsq['good_health'] = hsq['HSD010'].isin([1,2,3]).astype(int)

paq = pd.read_sas(BASE + 'PAQ_I.XPT', format='xport')[['SEQN','PAQ650']]  # vigorous activity y/n
smq = pd.read_sas(BASE + 'SMQ_I.XPT', format='xport')[['SEQN','SMQ020']]  # ever smoked 100 cigarettes
huq = pd.read_sas(BASE + 'HUQ_I.XPT', format='xport')[['SEQN','HUQ051']]  # # doctor visits past year

df = demo.merge(statin_flag, on='SEQN', how='left').merge(hsq, on='SEQN', how='inner') \
         .merge(paq, on='SEQN', how='left').merge(smq, on='SEQN', how='left') \
         .merge(huq, on='SEQN', how='left')

df['statin_user'] = df['statin_user'].fillna(False).astype(int)


df = df[df['RIDAGEYR'] >= 40].copy()
df.head()


,SEQN,RIDAGEYR,RIAGENDR,INDFMPIR,statin_user,HSD010,good_health,PAQ650,SMQ020,HUQ051
0,83732.0,62.0,1.0,4.39,0,3.0,1,2.0,1.0,5.000000e+00
1,83733.0,53.0,1.0,1.32,0,2.0,1,2.0,1.0,5.397605e-79
2,83734.0,78.0,1.0,1.51,0,4.0,0,2.0,1.0,2.000000e+00
3,83735.0,56.0,2.0,5.00,0,3.0,1,2.0,2.0,4.000000e+00
4,83736.0,42.0,2.0,1.23,0,4.0,0,2.0,2.0,2.000000e+00


## 2. Start with the raw comparison

Before adjusting for anything, what does the difference between statin users and non-users actually look like?

I first compared the proportion of people in each group reporting good health. I also looked at the characteristics that might make the two groups different in the first place.


In [ ]:
# Compare the proportion reporting good health, then test the raw difference.
raw_counts = pd.crosstab(df['statin_user'], df['good_health']).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

nonuser_good = raw_counts.loc[0, 1]
nonuser_total = raw_counts.loc[0].sum()
statin_good = raw_counts.loc[1, 1]
statin_total = raw_counts.loc[1].sum()

nonuser_prop = nonuser_good / nonuser_total
statin_prop = statin_good / statin_total
raw_difference = statin_prop - nonuser_prop

raw_comparison = pd.DataFrame({
    'Non-users': [nonuser_prop],
    'Statin users': [statin_prop],
    'Difference': [raw_difference]
}, index=['Good health'])

chi2, p_value, dof, expected = chi2_contingency(raw_counts, correction=False)

print("Raw comparison — share reporting good health:")
display(raw_comparison)

print(f"Chi-square test: χ² = {chi2:.2f}, p = {p_value:.4f}")

# Compare the measured characteristics that may explain differences between groups.
covariates_for_check = ['RIDAGEYR','RIAGENDR','INDFMPIR','PAQ650','SMQ020','HUQ051']
covariate_check = df.groupby('statin_user')[covariates_for_check].mean().T

label_map = {
    'RIDAGEYR': 'Age (years)',
    'RIAGENDR': 'Sex',
    'INDFMPIR': 'Income-to-poverty ratio',
    'PAQ650': 'Vigorous physical activity',
    'SMQ020': 'Ever smoked ≥100 cigarettes',
    'HUQ051': 'Doctor visits in past year'
}
covariate_check.index = covariate_check.index.map(label_map)

print("\nCovariate averages, by statin use:")
display(covariate_check)


Raw comparison — share reporting good health:


,Non-users,Statin users,Difference
Good health,0.671845,0.621256,-0.050589


Chi-square test: χ² = 8.39, p = 0.0038

Covariate averages, by statin use:


statin_user,0,1
Age (years),56.876117,66.823188
Sex,1.548350,1.446377
Income-to-poverty ratio,2.503737,2.447726
Vigorous physical activity,1.808932,1.899517
Ever smoked ≥100 cigarettes,1.575146,1.509179
Doctor visits in past year,2.551068,3.823188


The raw comparison gives us a starting point, but it is not yet a clean comparison. If statin users are systematically different from non-users, some of the difference in health could simply reflect those underlying differences. In this data, age and healthcare use are particularly important examples: people taking statins are older on average and have more doctor visits.

The next step is therefore to construct a comparison group that looks more like the statin users on the characteristics we measured.


## 3. Use propensity score matching to make the comparison more comparable

I first estimated each person's probability of being a statin user based on age, sex, income, physical activity, smoking history, and doctor visits. I then matched each statin user to a non-user with a similar estimated probability. This gives a more comparable set of groups: rather than comparing all statin users with all non-users, we are comparing statin users with non-users who look similar on the characteristics included in the model.


In [ ]:
covs = ['RIDAGEYR','RIAGENDR','INDFMPIR','PAQ650','SMQ020','HUQ051']
model_df = df.dropna(subset=covs + ['good_health']).copy()

X = model_df[covs].copy()
y = model_df['statin_user']
X = X.fillna(X.median())

ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X, y)
model_df['propensity'] = ps_model.predict_proba(X)[:, 1]

treated = model_df[model_df['statin_user'] == 1]
control = model_df[model_df['statin_user'] == 0]

nn = NearestNeighbors(n_neighbors=1).fit(control[['propensity']])
distances, indices = nn.kneighbors(treated[['propensity']])

matched_control = control.iloc[indices.flatten()].copy()
matched_treated = treated.copy()

# Apply the 0.05 caliper before calculating the outcome comparison.
keep = distances.flatten() <= 0.05
matched_treated = matched_treated.loc[keep].reset_index(drop=True)
matched_control = matched_control.loc[keep].reset_index(drop=True)

print(f"Matched pairs retained: {len(matched_treated)}")
print(f"Statin users excluded for propensity-score distance > 0.05: {(~keep).sum()}")

adjusted_health = pd.Series({
    'statin_user (matched)': matched_treated['good_health'].mean(),
    'non-user (matched control)': matched_control['good_health'].mean()
})

print("\nMatched comparison — share reporting good health:")
print(adjusted_health)


Matched pairs retained: 919
Statin users excluded for propensity-score distance > 0.05: 0

Matched comparison — share reporting good health:
statin_user (matched)         0.620239
non-user (matched control)    0.659412
dtype: float64


## 4. Did matching make the groups more similar?

The matching only helps if it actually makes the groups more similar. Here I compare the measured characteristics before and after matching. After matching, the statin and non-statin groups should look much more alike on the factors we used to construct the comparison.


In [ ]:
# Use reader-friendly labels rather than NHANES variable codes.
label_map = {
    'RIDAGEYR': 'Age (years)',
    'RIAGENDR': 'Sex',
    'INDFMPIR': 'Income-to-poverty ratio',
    'PAQ650': 'Vigorous physical activity',
    'SMQ020': 'Ever smoked ≥100 cigarettes',
    'HUQ051': 'Doctor visits in past year'
}

# For binary variables, report the proportion coded "yes" rather than the raw code mean.
before = df.groupby('statin_user')
after_t = matched_treated
after_c = matched_control

balance_before = pd.DataFrame({
    'Non-users (before)': [
        (df.loc[df.statin_user==0, 'RIDAGEYR']).mean(),
        (df.loc[df.statin_user==0, 'RIAGENDR']==2).mean(),
        (df.loc[df.statin_user==0, 'INDFMPIR']).mean(),
        (df.loc[df.statin_user==0, 'PAQ650']==1).mean(),
        (df.loc[df.statin_user==0, 'SMQ020']==1).mean(),
        (df.loc[df.statin_user==0, 'HUQ051']).mean()
    ],
    'Statin users (before)': [
        (df.loc[df.statin_user==1, 'RIDAGEYR']).mean(),
        (df.loc[df.statin_user==1, 'RIAGENDR']==2).mean(),
        (df.loc[df.statin_user==1, 'INDFMPIR']).mean(),
        (df.loc[df.statin_user==1, 'PAQ650']==1).mean(),
        (df.loc[df.statin_user==1, 'SMQ020']==1).mean(),
        (df.loc[df.statin_user==1, 'HUQ051']).mean()
    ]
}, index=list(label_map.values()))

balance_after = pd.DataFrame({
    'Statin users (matched)': [
        after_t['RIDAGEYR'].mean(),
        (after_t['RIAGENDR']==2).mean(),
        after_t['INDFMPIR'].mean(),
        (after_t['PAQ650']==1).mean(),
        (after_t['SMQ020']==1).mean(),
        after_t['HUQ051'].mean()
    ],
    'Non-users (matched)': [
        after_c['RIDAGEYR'].mean(),
        (after_c['RIAGENDR']==2).mean(),
        after_c['INDFMPIR'].mean(),
        (after_c['PAQ650']==1).mean(),
        (after_c['SMQ020']==1).mean(),
        after_c['HUQ051'].mean()
    ]
}, index=list(label_map.values()))

balance_comparison = pd.concat([balance_before, balance_after], axis=1)

print("Covariate balance before and after matching:")
display(balance_comparison)


Covariate balance before and after matching:


,Non-users (before),Statin users (before),Statin users (matched),Non-users (matched)
Age (years),56.876117,66.823188,66.698585,66.803047
Sex,0.548350,0.446377,0.430903,0.411317
Income-to-poverty ratio,2.503737,2.447726,2.447726,2.466148
Vigorous physical activity,0.193786,0.100483,0.094668,0.088139
Ever smoked ≥100 cigarettes,0.437670,0.504348,0.510337,0.515778
Doctor visits in past year,2.551068,3.823188,3.661589,3.064200


After matching, the groups are much closer on the measured characteristics. That gives us a better comparison than the original raw data. But it is important to distinguish more comparable from causally equivalent. Matching cannot account for factors that were not measured, and this dataset does not capture everything that could influence both statin use and health.


## 5. Does the health difference remain?

If the difference in health were largely explained by the measured differences between the groups, we would expect it to shrink after matching.

In [ ]:
# Compare self-rated health in the matched groups.
matched_health = pd.DataFrame({
    'Statin users (matched)': [matched_treated['good_health'].mean()],
    'Non-users (matched)': [matched_control['good_health'].mean()]
}, index=['Good health'])

matched_health['Difference'] = (
    matched_health['Statin users (matched)']
    - matched_health['Non-users (matched)']
)

# McNemar's test uses the paired outcomes created by the matching.
discordant_treated_good = ((matched_treated['good_health'] == 1) & (matched_control['good_health'] == 0)).sum()
discordant_control_good = ((matched_treated['good_health'] == 0) & (matched_control['good_health'] == 1)).sum()

mcnemar_table = np.array([
    [
        ((matched_treated['good_health'] == 1) & (matched_control['good_health'] == 1)).sum(),
        discordant_treated_good
    ],
    [
        discordant_control_good,
        ((matched_treated['good_health'] == 0) & (matched_control['good_health'] == 0)).sum()
    ]
])

mcnemar_result = mcnemar(mcnemar_table, exact=True)

print("Matched comparison — share reporting good health:")
display(matched_health)

print(f"McNemar's exact test: p = {mcnemar_result.pvalue:.4f}")


Matched comparison — share reporting good health:


,Statin users (matched),Non-users (matched),Difference
Good health,0.620239,0.659412,-0.039173


McNemar's exact test: p = 0.0921


The difference in reported good health did shrink when users were matched. However, the difference is small and not statistically significant.


## So What Should We Do With This Evidence?

The matched comparison shows a small difference in clinical outcome but does not justify building the feature. Additionally, the data is observational rather than experimental so the conclusions that can be drawn are limited.

For a stronger answer, a cohort design could follow patients from the point they start statins. That would establish the timing more clearly and make it possible to study adherence alongside subsequent outcomes. The analysis would also benefit from more objective clinical outcome measures such as biomarkers.  

There is also a product question underneath the statistical one: **why are patients missing their medication?** Forgetting is only one possibility. There are many possible reasons for non-adherence to medication. If the real barrier is cost, side effects, or concerns about the medication, a reminder feature may be solving the wrong problem, which could result in churn or low engagement.


## Recommendations

**Do not add the feature yet.** These findings alone do not yet indicate there would be a benefit of a reminder feature. A further study, such as a cohort analysis, could better illuminate the relationship between statins and health outcomes.

**Check the feature design.** First identify the main barriers to medication adherence in this population. A framework such as the COM-B model of behavior change can help identify the actual barriers around medication adherence.

**Then run a small experiment.** If forgetting turns out to be the main barrier, an A/B test of reminders or refill nudges could be implemented if the feature is justifed by further analyses. Otherwise, the A/B test should focus on a feature addressing the identified barrier.

